# 과제 - PyTorch로 mini GPT 구현

이 노트북은 과제 안내서의 구현 순서를 그대로 따릅니다.

1. 환경설정
2. NSMC 데이터 준비
3. BPE 토크나이저
4. GPTDataset / InputEmbedding
5. MultiHeadAttention
6. GPTModel
7. 사전 학습 유틸리티
8. 감성 분류 미세 조정

각 단계의 `src/` TODO를 구현한 뒤, 바로 아래 pytest 셀로 해당 단계만 확인하세요.

## 1. 환경설정

Colab에서는 GitHub 저장소 URL과 GitHub Personal Access Token을 입력해 저장소를 clone하고 `src/`를 import 경로에 추가합니다.
로컬 VS Code에서는 현재 폴더를 프로젝트 루트로 보고 실행합니다.

In [ ]:
# Colab: 이 셀을 가장 먼저 실행하세요.
import os
import subprocess
import sys
from pathlib import Path


def normalize_github_url(url: str) -> str:
    """Colab 입력값을 git clone에 사용할 수 있는 https URL로 정리합니다."""
    url = url.strip()
    if not url:
        raise ValueError("GitHub 저장소 URL을 입력해야 합니다.")
    if url.startswith("github.com/"):
        url = "https://" + url
    if not url.startswith("https://"):
        raise ValueError("저장소 URL은 https://github.com/... 또는 github.com/... 형식이어야 합니다.")
    url = url.rstrip("/")
    if not url.endswith(".git"):
        url += ".git"
    return url


if "google.colab" in sys.modules:
    from getpass import getpass

    repo_url = normalize_github_url(input("GitHub 저장소 URL (예: github.com/USERNAME/gpt-lab.git): "))
    token = getpass("GitHub Personal Access Token (Private 저장소인 경우 입력, 공개 저장소면 Enter): ").strip()
    clone_url = repo_url.replace("https://", f"https://{token}@") if token else repo_url
    repo_name = Path(repo_url[:-4]).name if repo_url.endswith(".git") else Path(repo_url).name
    repo_dir = Path("/content") / repo_name

    if not repo_dir.exists():
        subprocess.run(["git", "clone", clone_url, str(repo_dir)], check=True)
        subprocess.run(["git", "remote", "set-url", "origin", repo_url], cwd=repo_dir, check=True)
    else:
        print(f"이미 clone된 저장소를 사용합니다: {repo_dir}")

    os.chdir(repo_dir)
else:
    repo_dir = Path(".").resolve()

sys.path.insert(0, str(repo_dir / "src"))
print(f"Repo: {repo_dir}")

In [ ]:
# 단계별 테스트 실행 helper
import subprocess
import sys


def run_pytest(target: str):
    cmd = [sys.executable, "-m", "pytest", target, "-v"]
    print("실행 명령:", " ".join(cmd))
    result = subprocess.run(cmd, cwd=str(repo_dir), text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        print("\n아직 통과하지 못한 테스트가 있습니다. 해당 단계의 TODO를 먼저 구현하세요.")
    else:
        print("\n선택한 테스트를 통과했습니다.")
    return result.returncode

## 2. NSMC 데이터 준비

기본 데이터는 NAVER Sentiment Movie Corpus(NSMC)입니다.
`download_data.py`는 원본 TSV를 내려받고, 사전 학습용 텍스트와 감성 분류용 JSONL을 만듭니다.

In [ ]:
from pathlib import Path

try:
    import download_data

    paths = download_data.main()
except Exception as e:
    print("데이터 준비 중 문제가 생겼습니다:", e)
    print("이미 data/ 파일이 있다면 다음 셀부터 계속 진행할 수 있습니다.")

LM_TRAIN_PATH = repo_dir / "data" / "nsmc_lm_train.txt"
LM_VAL_PATH = repo_dir / "data" / "nsmc_lm_val.txt"
print("LM train exists:", LM_TRAIN_PATH.exists(), LM_TRAIN_PATH)
print("LM val exists:", LM_VAL_PATH.exists(), LM_VAL_PATH)

In [ ]:
corpus = LM_TRAIN_PATH.read_text(encoding="utf-8") if LM_TRAIN_PATH.exists() else ""
val_corpus = LM_VAL_PATH.read_text(encoding="utf-8") if LM_VAL_PATH.exists() else ""
print("train chars:", len(corpus))
print("val chars:", len(val_corpus))
print(corpus[:200])

## 3. Step 1 - BPE 토크나이저

구현 파일: `src/bpe.py`

먼저 `pytest tests/test_bpe.py -v`를 통과시키세요. 한국어를 안전하게 다루기 위해 UTF-8 byte-level BPE로 구현해야 합니다.

In [ ]:
run_pytest("tests/test_bpe.py")

In [ ]:
# BPE 구현 후 작은 말뭉치로 인코딩/디코딩 복원을 확인합니다.
try:
    from bpe import BPETokenizer

    tokenizer = BPETokenizer(vocab_size=300)
    tokenizer.train(corpus[:5000])
    sample = "이 영화는 정말 좋았다! English 123"
    ids = tokenizer.encode(sample, add_bos_eos=True)
    print(ids[:20])
    print(tokenizer.decode(ids))
except NotImplementedError as e:
    print("BPE TODO 미구현:", e)

## 4. Step 2 - GPTDataset / InputEmbedding

구현 파일: `src/dataset.py`, `src/embeddings.py`

BPE가 통과한 뒤 데이터셋과 입력 임베딩을 구현합니다.

In [ ]:
run_pytest("tests/test_dataset.py")

In [ ]:
try:
    from bpe import BPETokenizer
    from dataset import create_dataloader
    from embeddings import InputEmbedding

    tokenizer = BPETokenizer(vocab_size=300)
    tokenizer.train(corpus[:5000])
    token_ids = tokenizer.encode(corpus[:5000])
    loader = create_dataloader(token_ids, context_length=32, batch_size=2, shuffle=False)
    inp, tgt = next(iter(loader))
    emb = InputEmbedding(vocab_size=300, emb_dim=32, context_length=32, drop_rate=0.0)
    out = emb(inp)
    print(inp.shape, tgt.shape, out.shape)
except NotImplementedError as e:
    print("Dataset/Embedding TODO 미구현:", e)

## 5. Step 3 - MultiHeadAttention

구현 파일: `src/attention.py`

Q/K/V shape, head 분리, causal mask를 차례로 확인하세요.

In [ ]:
run_pytest("tests/test_attention.py")

## 6. Step 4 - GPTModel

구현 파일: `src/model.py`

LayerNorm, GELU, FeedForward, TransformerBlock, GPTModel, `generate_text_simple` 순서로 구현합니다.

In [ ]:
run_pytest("tests/test_model.py")

In [ ]:
try:
    import torch
    from model import GPTModel

    config = {
        "vocab_size": 300,
        "context_length": 32,
        "emb_dim": 32,
        "n_heads": 4,
        "n_layers": 1,
        "drop_rate": 0.0,
        "qkv_bias": False,
    }
    model = GPTModel(config)
    x = torch.randint(0, config["vocab_size"], (2, 16))
    logits = model(x)
    print(logits.shape)
except NotImplementedError as e:
    print("Model TODO 미구현:", e)

## 7. Step 5 - 사전 학습 유틸리티

구현 파일: `src/train.py`

loss 계산, checkpoint 저장/로드, temperature/top-k 생성, `train_model`을 구현합니다.

In [ ]:
run_pytest("tests/test_train.py")

In [ ]:
# 모든 앞 단계가 구현된 뒤 한 배치 smoke test를 실행합니다.
try:
    import torch
    from bpe import BPETokenizer
    from dataset import create_dataloader
    from model import GPTModel
    from train import calc_loss_batch

    tokenizer = BPETokenizer(vocab_size=300)
    tokenizer.train(corpus[:5000])
    token_ids = tokenizer.encode(corpus[:5000])
    loader = create_dataloader(token_ids, context_length=32, batch_size=2, shuffle=False)
    inp, tgt = next(iter(loader))
    config = {
        "vocab_size": 300,
        "context_length": 32,
        "emb_dim": 32,
        "n_heads": 4,
        "n_layers": 1,
        "drop_rate": 0.0,
        "qkv_bias": False,
    }
    model = GPTModel(config)
    loss = calc_loss_batch(inp, tgt, model, torch.device("cpu"))
    loss.backward()
    print("smoke loss:", loss.item())
except NotImplementedError as e:
    print("사전 학습 TODO 미구현:", e)

## 8. Step 6 - 감성 분류 미세 조정

구현 파일: `src/finetune.py`

NSMC JSONL/TSV를 읽어 분류 Dataset을 만들고, GPT backbone 위에 classification head를 붙입니다.

In [ ]:
run_pytest("tests/test_finetune.py")

## 9. 전체 테스트와 제출 전 확인

각 단계 테스트가 모두 통과하면 마지막에 전체 테스트를 실행합니다.

In [ ]:
run_pytest("tests/")

In [ ]:
# Colab: 현재 실험 브랜치를 clone/checkout합니다.
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Jungle-12-303/week14-team-05-gpt-lab.git"
BRANCH = "experiment/d0-baseline-20epoch"
REPO_DIR = Path("/content/week14-team-05-gpt-lab")


def run(cmd, cwd=None):
    print("$", " ".join(map(str, cmd)), flush=True)
    subprocess.run(list(map(str, cmd)), cwd=cwd, check=True)


if not REPO_DIR.exists():
    run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR])
else:
    run(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR)
    run(["git", "checkout", BRANCH], cwd=REPO_DIR)
    run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=REPO_DIR)

%cd /content/week14-team-05-gpt-lab


In [ ]:
# Colab/로컬: GitHub에서 pull된 A0 archive를 런타임에 압축 해제합니다.
import shutil
import tarfile
from pathlib import Path

repo_dir = Path("/content/week14-team-05-gpt-lab") if Path("/content/week14-team-05-gpt-lab").exists() else Path.cwd()
pretrain_root = repo_dir / "local" / "experiment_outputs" / "pretrain"
target_dir = pretrain_root / "A0_20260602_JAEHWAN"
archive_path = repo_dir / "A0_20260602_JAEHWAN.tar.gz"

if not archive_path.exists():
    raise FileNotFoundError(
        f"A0 archive not found: {archive_path}\n"
        "먼저 clone/checkout 셀을 다시 실행해서 GitHub의 현재 브랜치를 pull 하세요."
    )

pretrain_root.mkdir(parents=True, exist_ok=True)
if target_dir.exists():
    shutil.rmtree(target_dir)
with tarfile.open(archive_path, "r:gz") as tar:
    tar.extractall(pretrain_root)

required_files = [
    target_dir / "summary.json",
    target_dir / "checkpoints" / "A0_20260602_step0040_best.pt",
]
missing = [path for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required A0 files after extract:\n" + "\n".join(map(str, missing)))

print("A0 pretrain result is ready in runtime:", target_dir)
print("summary:", target_dir / "summary.json")
print("best checkpoint:", target_dir / "checkpoints" / "A0_20260602_step0040_best.pt")


In [ ]:
# Colab/로컬: A0 pretrain 결과를 Google Drive로 복사합니다.
import shutil
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    from google.colab import drive

    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")

repo_dir = Path("/content/week14-team-05-gpt-lab") if Path("/content/week14-team-05-gpt-lab").exists() else Path.cwd()
source_dir = repo_dir / "local" / "experiment_outputs" / "pretrain" / "A0_20260602_JAEHWAN"
drive_pretrain_root = Path("/content/drive/MyDrive/gpt-lab/experiment_outputs/pretrain")
target_dir = drive_pretrain_root / source_dir.name

if not source_dir.exists():
    raise FileNotFoundError(
        f"A0 source directory not found: {source_dir}\n"
        "Colab에서 실행 중이라면 로컬 PC의 local/ 폴더는 자동으로 보이지 않습니다. "
        "이 폴더를 먼저 Colab 런타임에 업로드하거나, Git/Drive에 올린 뒤 다시 실행하세요."
    )

required_files = [
    source_dir / "summary.json",
    source_dir / "checkpoints" / "A0_20260602_step0040_best.pt",
]
missing = [path for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required A0 files:\n" + "\n".join(map(str, missing)))

drive_pretrain_root.mkdir(parents=True, exist_ok=True)
if target_dir.exists():
    shutil.rmtree(target_dir)
shutil.copytree(source_dir, target_dir)

print("copied A0 pretrain result to:", target_dir)
print("summary:", target_dir / "summary.json")
print("best checkpoint:", target_dir / "checkpoints" / "A0_20260602_step0040_best.pt")
print("\n다음 D0 셀은 이 경로를 자동으로 찾거나, 아래 환경변수로 직접 지정할 수 있습니다:")
print(f'os.environ["GPT_LAB_PRETRAIN_RUN_DIR"] = "{target_dir}"')


In [ ]:
# D0 baseline: 10 epochs, batch size 64 (Colab/Drive run)
import os
import subprocess
import sys
from datetime import datetime
from pathlib import Path

BRANCH = os.environ.get("GPT_LAB_BRANCH", "experiment/d0-baseline-20epoch")
REPO_URL = os.environ.get(
    "GPT_LAB_REPO_URL",
    "https://github.com/Jungle-12-303/week14-team-05-gpt-lab.git",
)
REPO_DIR = Path(os.environ.get("GPT_LAB_REPO_DIR", "/content/week14-team-05-gpt-lab"))
DRIVE_ROOT = Path(os.environ.get("GPT_LAB_DRIVE_ROOT", "/content/drive/MyDrive/gpt-lab"))
PRETRAIN_ROOT = DRIVE_ROOT / "experiment_outputs" / "pretrain"
SENTIMENT_ROOT = DRIVE_ROOT / "experiment_outputs" / "sentiment"


def run(cmd, cwd=None):
    print("$", " ".join(map(str, cmd)), flush=True)
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    subprocess.run(list(map(str, cmd)), cwd=cwd, check=True, env=env)


if "google.colab" in sys.modules:
    from google.colab import drive

    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")

if not REPO_DIR.exists():
    run(["git", "clone", REPO_URL, REPO_DIR])
run(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR)
run(["git", "checkout", BRANCH], cwd=REPO_DIR)
run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=REPO_DIR)
run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=REPO_DIR)

try:
    import torch
except ModuleNotFoundError as exc:
    raise RuntimeError("PyTorch is not installed after requirements installation.") from exc

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required. In Colab, choose Runtime > Change runtime type > GPU.")
print("cuda:", torch.cuda.get_device_name(0), flush=True)

def is_valid_pretrain_run(path):
    return (path / "summary.json").exists() and any((path / "checkpoints").glob("*_best.pt"))


def find_pretrain_run_dir():
    explicit = os.environ.get("GPT_LAB_PRETRAIN_RUN_DIR")
    if explicit:
        path = Path(explicit)
        if is_valid_pretrain_run(path):
            return path
        raise FileNotFoundError(
            f"GPT_LAB_PRETRAIN_RUN_DIR is not a valid pretrain run directory: {path}\n"
            "Expected summary.json and checkpoints/*_best.pt inside it."
        )

    search_roots = [
        PRETRAIN_ROOT,
        DRIVE_ROOT,
        Path("/content/drive/MyDrive"),
        REPO_DIR / "local" / "experiment_outputs" / "pretrain",
        Path.cwd() / "local" / "experiment_outputs" / "pretrain",
    ]
    patterns = ["A0_basic_*", "A0_*", "**/A0_basic_*", "**/A0_*"]
    seen = set()
    candidates = []
    inspected = []

    for root in search_roots:
        if not root.exists():
            inspected.append(f"missing root: {root}")
            continue
        for pattern in patterns:
            for candidate in sorted(root.glob(pattern), reverse=True):
                if candidate in seen or not candidate.is_dir():
                    continue
                seen.add(candidate)
                has_summary = (candidate / "summary.json").exists()
                best_count = len(list((candidate / "checkpoints").glob("*_best.pt")))
                inspected.append(f"{candidate} (summary={has_summary}, best_checkpoints={best_count})")
                if has_summary and best_count > 0:
                    candidates.append(candidate)

    if candidates:
        print("pretrain candidates:", flush=True)
        for candidate in candidates[:5]:
            print("-", candidate, flush=True)
        return candidates[0]

    print("searched pretrain locations:", flush=True)
    for item in inspected[:30]:
        print("-", item, flush=True)
    raise FileNotFoundError(
        "No valid A0 pretrain run found. "
        "Check that Google Drive is mounted with the account that contains the A0 result, "
        "or set GPT_LAB_PRETRAIN_RUN_DIR to the exact folder containing summary.json and checkpoints/*_best.pt."
    )


pretrain_run_dir = find_pretrain_run_dir()

run_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = SENTIMENT_ROOT / f"D0_baseline_10epoch_bs64_{run_stamp}_HYEONGMIN"
output_dir.mkdir(parents=True, exist_ok=True)

print("pretrain_run_dir:", pretrain_run_dir, flush=True)
print("output_dir:", output_dir, flush=True)

run(
    [
        sys.executable,
        "experiments/scripts/run_d_sentiment.py",
        "--experiment",
        "D0",
        "--epochs",
        "10",
        "--batch-size",
        "64",
        "--pretrain-run-dir",
        pretrain_run_dir,
        "--output-dir",
        output_dir,
        "--device",
        "cuda",
        "--require-cuda",
        "--num-workers",
        "2",
        "--pin-memory",
        "--persistent-workers",
        "--prefetch-factor",
        "2",
        "--enable-tf32",
        "--log-every-steps",
        "20",
    ],
    cwd=REPO_DIR,
)

print("D0 10-epoch batch-size-64 run complete.")
print("summary:", output_dir / "summary.json")
print("metrics:", output_dir / "metrics")
print("checkpoints:", output_dir / "checkpoints")
print("logs:", output_dir / "logs")
